In [33]:
import pandas as pd                                   # load the table library
df = pd.read_csv("uxbridge_lounge_sales_raw.csv")     # read the raw file (read only, never overwrite)
print(df.shape)                                       # how many rows and columns
df.info()                                             # column types and how many blanks each has
df.head(10)                                           # show the first 10 rows

(13342, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13342 entries, 0 to 13341
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   record_id      13342 non-null  int64  
 1   date           13342 non-null  object 
 2   day_of_week    13342 non-null  object 
 3   item_name      13342 non-null  object 
 4   category       13342 non-null  object 
 5   cuisine        13342 non-null  object 
 6   price_gbp      13342 non-null  object 
 7   price_tier     13342 non-null  object 
 8   unit_cost_gbp  13342 non-null  float64
 9   shelf_life     13342 non-null  object 
 10  weather        12940 non-null  object 
 11  temperature_c  12778 non-null  object 
 12  festival       3917 non-null   object 
 13  local_event    587 non-null    object 
 14  prepared_qty   13342 non-null  int64  
 15  sold_qty       13143 non-null  float64
 16  wasted_qty     13342 non-null  int64  
 17  stockout       13342 non-null  object 

,record_id,date,day_of_week,item_name,category,cuisine,price_gbp,price_tier,unit_cost_gbp,shelf_life,weather,temperature_c,festival,local_event,prepared_qty,sold_qty,wasted_qty,stockout
0,105341,11 Aug 2025,Monday,Mango Lassi,Drink,Punjabi,4.5,Low,1.1,Same day,Cloudy,18.5,none,NaN,19,19.0,0,Yes
1,101369,2025-02-27,Thursday,Butter Chicken,Food,Punjabi,13.95,High,4.2,Same day,Cloudy,9.5,NaN,NaN,23,23.0,0,No
2,110106,2026-02-27,Friday,Chaas,Drink,Gujarati,3.5,Low,0.5,Same day,Sunny,2.5,NaN,NaN,14,14.0,0,Yes
3,103546,2025-05-28,Wednesday,Paneer Tikka,Food,Punjabi,9.5,Moderate,3.1,Same day,Rainy,12.6,-,NaN,15,11.0,4,No
4,105467,16/08/2025,Saturday,Pav Bhaji,Food,Marathi,£9.00,Moderate,1.8,Same day,Sunny,23.6,No festival,NaN,24,24.0,0,No
5,111334,2026-04-19,Sunday,Dal Makhani,Food,Punjabi,10.5,High,2.1,Same day,Cloudy,9.1,NaN,NaN,22,22.0,0,Yes
6,109250,2026-01-22,Thursday,Idli Sambar,Food,South Indian,7.5,Moderate,1.2,Same day,Cloudy,2.6,NaN,NaN,13,11.0,2,No
7,102895,2025-05-01,Thursday,Masala Dosa,Food,South Indian,9.95,Moderate,1.9,Same day,Cloudy,9.9,No festival,India cricket match,20,19.0,1,No
8,103636,2025-06-01,Sunday,Kingfisher Lager,Drink,Bar,5.5,Low,1.6,Long,Sunny,12.9,NaN,NaN,79,33.0,1,No
9,108356,2025-12-15,Monday,Chole Bhature,Food,Punjabi,11.0,High,2.4,Same day,snow,1.3,NaN,NaN,11,11.0,0,No


price_gbp → should be a number (money) → shows object ❌
temperature_c → should be a number (degrees) → shows object ❌
date → should be a date → shows object ❌
Sold_qty is float64 (decimals), but you can’t sell half a dosa. Blanks force pandas to use decimals.
: total rows = 13,342. Any column with a smaller “Non-Null Count” has blanks.

weather 12,940 → 402 blanks
temperature_c 12,778 → 564 blanks
sold_qty 13,143 → 199 blanks
festival 3,917 → 9,425 blanks
local_event 587 → 12,755 blanks

Not all blanks are the same!
✔ weather, temperature, sold_qty: truly missing, something went wrong
✔ festival, local_event: most blanks just mean “no festival / no event that day”. Not an error, just badly recorded



In [34]:
text_cols = ["day_of_week", "item_name", "category", "cuisine",
             "price_tier", "shelf_life", "weather", "festival",
             "local_event", "stockout"]                          # the columns that hold words

for col in text_cols:                                            # go through them one by one
    print(f"\n===== {col} =====")                                # print a heading for each column
    print(df[col].value_counts(dropna=False))                    # count each unique value, including blanks (NaN)


===== day_of_week =====
day_of_week
Wednesday    1919
Friday       1911
Saturday     1910
Tuesday      1910
Monday       1909
Sunday       1900
Thursday     1883
Name: count, dtype: int64

===== item_name =====
item_name
Sol Kadhi                 539
Paneer Tikka              534
Chicken Chettinad         534
Cobra Beer                534
Dhokla                    533
                         ... 
  Thepla with Chutney       3
FILTER COFFEE               3
COBRA BEER                  3
IDLI SAMBAR                 2
SOL KADHI                   2
Name: count, Length: 104, dtype: int64

===== category =====
category
Food      8544
Drink     4266
 Food      125
food       119
FOOD       108
drink       65
DRINK       59
Drinks      56
Name: count, dtype: int64

===== cuisine =====
cuisine
Punjabi         3339
South Indian    2789
Marathi         2777
Gujarati        2769
Bar             1668
Name: count, dtype: int64

===== price_tier =====
price_tier
Moderate    6120
Low         4442
Hig

1. Item Name: This is the biggest issue! The bottom of the list shows a total of 104 unique entries, even though we only have 24 actual menu items. That means there are 80 extra variations caused by inconsistent naming. For example, some entries have unnecessary spaces at the beginning, such as " Thepla with Chutney", while others have different spellings or naming formats, like "COBRA BEER" and "SOL KADHI".

2. Category: We should only have 2 categories: Food and Drinks. However, the dataset contains 8 variations. For example, " Food" has an extra space at the beginning, while "Drinks" has an additional 's'. These small inconsistencies make Python treat them as separate categories.

3. Weather: We should have only 4 weather conditions, but the dataset contains 18 different entries. There are a couple of hidden problems here:
- "Rain" is used instead of the correct category, "Rainy".
- "Cloudy" appears twice because one version has an extra space at the end.
These are good examples of how seemingly minor data-entry mistakes can create major problems during data analysis.

In [35]:
num_cols = ["unit_cost_gbp", "prepared_qty", "sold_qty", "wasted_qty"]  # columns that are already numbers
print(df[num_cols].describe().round(1))                              # min, max, average etc. for each

temp_check = pd.to_numeric(df["temperature_c"], errors="coerce")      # try turning temperature into numbers (bad text becomes blank)
print(temp_check.describe().round(1))                                # summary of temperature

bad_text = df.loc[temp_check.isna() & df["temperature_c"].notna(), "temperature_c"]  # rows that had text, not numbers
print(bad_text.value_counts())                                       # which text values were in there

       unit_cost_gbp  prepared_qty  sold_qty  wasted_qty
count        13342.0       13342.0   13143.0     13342.0
mean             1.7          19.8      16.1         1.1
std              1.0          17.2      13.9         1.7
min              0.4           4.0     -39.0        -9.0
25%              1.1          10.0       9.0         0.0
50%              1.6          15.0      14.0         0.0
75%              2.1          23.0      20.0         2.0
max              4.2         258.0     670.0        15.0
count    12686.0
mean        11.7
std          8.7
min         -2.0
25%          6.6
50%         10.3
75%         14.8
max         78.1
Name: temperature_c, dtype: float64
temperature_c
-          55
unknown    37
Name: count, dtype: int64


📊 PART 2: READING YOUR OUTPUT
━━━━━━━━━━━━━━━━━━
(std = how spread out the numbers are. Big std = values vary a lot. You can mostly ignore it for now.)

💷 unit_cost_gbp
min 0.4, max 4.2 → matches our menu (chai £0.40 to butter chicken £4.20) ✅ Clean

🍳 prepared_qty
min 4 ✅ possible
max 258 vs 75% = 23 🤔 looks crazy, BUT...
🧠 Abhishek asks “which item?” before calling it an error. Beer bottles are stocked generously, and on a busy cricket Saturday, chilling 258 Kingfishers is believable. Probably real. We’ll confirm later.

🛒 sold_qty
count 13,143 → 199 blanks (we knew this)
min -39 ❌ you can’t sell minus 39 dosas → negative values
max 670 ❌ and here’s Abhishek’s killer check: the most ever prepared is 258. You can’t sell 670 if you only made 258! → outliers (likely someone typed an extra zero: 67 became 670)

🗑️ wasted_qty
min -9 ❌ negative waste is impossible → negative values
max 15 ✅ believable

🌡️ temperature_c
min -2 ✅ a cold London night
max 78.1 ❌ London has never been 78°C! 78°F = 25.6°C → some rows recorded in Fahrenheit 🇺🇸
count 12,686 vs 13,342 rows → 656 non-numbers in total

📝 text in temperature
“-” 55 times, “unknown” 37 times → placeholder text ❌

🕵️ Bonus mystery: the generator also put “N/A” in there, but it’s not in your list. Why? Pandas silently turns “N/A” into blank when reading a CSV. Same happened to “None” in festival! Lesson: pandas does some cleaning without telling you, so Abhishek always documents it.





In [36]:
import pandas as pd

# Ensure df is defined by loading the raw file
df = pd.read_csv("uxbridge_lounge_sales_raw.csv")

# CHECK 1: duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())                   # rows that are 100% copies of another row
print("Duplicate record_ids:", df["record_id"].duplicated().sum())      # IDs that appear more than once (IDs should be unique)

# CHECK 2: are rows in date order?
print("Record IDs in order?", df["record_id"].is_monotonic_increasing)  # True = sorted, False = jumbled

# CHECK 3: does day_of_week match the date?
dates = (pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")             # try format 1: 2025-02-27
         .fillna(pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce"))   # fill gaps with format 2: 27/02/2025
         .fillna(pd.to_datetime(df["date"], format="%d %b %Y", errors="coerce")))  # fill gaps with format 3: 27 Feb 2025
print("Dates that failed to read:", dates.isna().sum())                 # should be 0
print("Date range:", dates.min(), "to", dates.max())
wrong_day = dates.dt.day_name() != df["day_of_week"]                    # work out the real day, compare with the written day
print("Rows with wrong day_of_week:", wrong_day.sum())

# CHECK 4: does waste add up? (fresh food only: prepared - sold should equal wasted)
fresh = df["shelf_life"] == "Same day"                                  # only freshly made items
bad_waste = fresh & df["sold_qty"].notna() & (df["wasted_qty"] != df["prepared_qty"] - df["sold_qty"])  # the sum doesn't add up
print("Rows where waste doesn't add up:", bad_waste.sum())

Exact duplicate rows: 262
Duplicate record_ids: 262
Record IDs in order? False
Dates that failed to read: 0
Date range: 2025-01-01 00:00:00 to 2026-06-30 00:00:00
Rows with wrong day_of_week: 65
Rows where waste doesn't add up: 238


In [37]:
clean = df.copy()                                    # make a working copy, raw df stays untouched
log = []                                             # empty list to record every fix we make

# FIX 1: remove duplicates
before = len(clean)                                  # count rows before
clean = clean.drop_duplicates()                      # delete rows that are exact copies
log.append(("Removed exact duplicate rows", before - len(clean)))  # write down how many were removed

# FIX 2: put all dates into one format
clean["date"] = (pd.to_datetime(clean["date"], format="%Y-%m-%d", errors="coerce")         # read 2025-02-27 style
                 .fillna(pd.to_datetime(clean["date"], format="%d/%m/%Y", errors="coerce"))  # then 27/02/2025 style
                 .fillna(pd.to_datetime(clean["date"], format="%d %b %Y", errors="coerce"))) # then 27 Feb 2025 style
assert clean["date"].isna().sum() == 0               # safety check: stop with an error if any date failed

# FIX 3: recalculate day_of_week from the date
real_day = clean["date"].dt.day_name()               # work out the true day from each date
wrong = (real_day != clean["day_of_week"]).sum()     # count how many were wrong
clean["day_of_week"] = real_day                      # replace the column with the true days
log.append(("Corrected day_of_week to match date", wrong))  # write it down

# FIX 4: sort into time order
clean = clean.sort_values(["date", "record_id"]).reset_index(drop=True)  # oldest first, then renumber the rows 0, 1, 2...
log.append(("Sorted rows by date", len(clean)))      # write it down

# CHECK: did it work?
print(pd.DataFrame(log, columns=["step", "rows_affected"]))       # show the cleaning log as a table
print("Shape:", clean.shape)                                      # rows and columns now
print("Date type:", clean["date"].dtype)                          # should say datetime, not object
print("IDs unique?", clean["record_id"].is_unique)                # should be True
print("In order?", clean["record_id"].is_monotonic_increasing)    # should be True

                                  step  rows_affected
0         Removed exact duplicate rows            262
1  Corrected day_of_week to match date             65
2                  Sorted rows by date          13080
Shape: (13080, 18)
Date type: datetime64[ns]
IDs unique? True
In order? True


In [38]:
# FIX 5 + 6: item names, matched to the official menu
MENU = ["Butter Chicken", "Dal Makhani", "Chole Bhature", "Paneer Tikka",          # Punjabi
        "Dhokla", "Undhiyu", "Thepla with Chutney", "Khandvi",                     # Gujarati
        "Vada Pav", "Misal Pav", "Pav Bhaji", "Puran Poli",                        # Marathi
        "Masala Dosa", "Idli Sambar", "Medu Vada", "Chicken Chettinad",            # South Indian
        "Mango Lassi", "Masala Chai", "Chaas", "Sol Kadhi", "Filter Coffee",       # soft drinks
        "Kingfisher Lager", "Cobra Beer", "House Red Wine 175ml"]                  # bar
item_fix = {name.lower(): name for name in MENU}                    # lookup: "butter chicken" -> "Butter Chicken"
item_fix.update({"butter chiken": "Butter Chicken", "masala dosaa": "Masala Dosa",  # add the typos we found
                 "chole bature": "Chole Bhature", "paav bhaji": "Pav Bhaji",
                 "idly sambar": "Idli Sambar", "dhoklaa": "Dhokla",
                 "kingfisher": "Kingfisher Lager"})
old = clean["item_name"].copy()                                     # keep the old values to count changes
clean["item_name"] = clean["item_name"].str.strip().str.lower().map(item_fix)   # strip, lower, translate
assert clean["item_name"].isna().sum() == 0                         # tripwire: every name must be on the menu
log.append(("Standardised item names to menu", (old != clean["item_name"]).sum()))

# FIX 7: category
old = clean["category"].copy()
clean["category"] = clean["category"].str.strip().str.lower().map({"food": "Food", "drink": "Drink", "drinks": "Drink"})
assert clean["category"].isna().sum() == 0                          # tripwire: only Food or Drink allowed
log.append(("Standardised category", (old != clean["category"]).sum()))

# FIX 8: weather (blanks stay blank for now, we fill them in Stage C)
old = clean["weather"].copy()
clean["weather"] = clean["weather"].str.strip().str.title().replace({"Rain": "Rainy"})  # title is safe here: single words
log.append(("Standardised weather spelling", ((old != clean["weather"]) & old.notna()).sum()))

# FIX 9: festival
clean["festival"] = (clean["festival"].fillna("None").str.strip()                       # blank -> "None"
                     .replace({"none": "None", "-": "None", "No festival": "None"}))     # other versions -> "None"
log.append(("Standardised 'no festival' to None", (clean["festival"] == "None").sum()))

# FIX 10: local_event
clean["local_event"] = clean["local_event"].fillna("None")          # blank means no event that day
log.append(("Filled blank local_event with None", (clean["local_event"] == "None").sum()))

# FIX 11: stockout
old = clean["stockout"].copy()
clean["stockout"] = clean["stockout"].str.strip().str.upper().map(
    {"YES": "Yes", "Y": "Yes", "TRUE": "Yes", "NO": "No", "N": "No", "FALSE": "No"})   # 8 versions -> 2
assert clean["stockout"].isna().sum() == 0                          # tripwire: only Yes or No allowed
log.append(("Standardised stockout to Yes/No", (old != clean["stockout"]).sum()))

# CHECK: how many unique values now?
for col in ["item_name", "category", "weather", "festival", "local_event", "stockout"]:
    print(col, "->", clean[col].nunique(dropna=False), "unique values")   # count groups, including blanks
print(pd.DataFrame(log, columns=["step", "rows_affected"]))               # show the full log so far

item_name -> 24 unique values
category -> 2 unique values
weather -> 5 unique values
festival -> 8 unique values
local_event -> 2 unique values
stockout -> 2 unique values
                                  step  rows_affected
0         Removed exact duplicate rows            262
1  Corrected day_of_week to match date             65
2                  Sorted rows by date          13080
3      Standardised item names to menu            654
4                Standardised category            523
5        Standardised weather spelling            763
6   Standardised 'no festival' to None          11952
7   Filled blank local_event with None          12504
8      Standardised stockout to Yes/No            261


In [39]:
# FIX 12: price, remove £ and make it a number
old = clean["price_gbp"].astype(str)                                        # look at prices as text
clean["price_gbp"] = pd.to_numeric(old.str.replace("£", "", regex=False))   # delete the £ sign, then convert to number
log.append(("Removed £ sign from price", old.str.contains("£").sum()))      # count how many had a £
assert clean.groupby("item_name")["price_gbp"].nunique().max() == 1         # tripwire: each item must have ONE price

# FIX 13: temperature, the team huddle 🗳️
temp = pd.to_numeric(clean["temperature_c"], errors="coerce")               # text like "unknown" or "-" becomes blank
day_temp = temp.groupby(clean["date"]).transform("median")                  # each day votes: middle value wins
log.append(("Fixed temperature (Fahrenheit, text, blanks) using daily median", (temp != day_temp).sum()))
clean["temperature_c"] = day_temp                                           # replace with the agreed daily value
assert clean["temperature_c"].isna().sum() == 0                             # tripwire: no blanks left
assert clean["temperature_c"].between(-10, 35).all()                        # tripwire: realistic London range

# FIX 14: weather blanks, same huddle idea
blanks = clean["weather"].isna().sum()                                                         # count blanks first
clean["weather"] = clean.groupby("date")["weather"].transform(lambda s: s.mode().iloc[0])      # most common weather that day wins
log.append(("Filled blank weather using most common weather that day", blanks))
assert clean["weather"].isna().sum() == 0                                   # tripwire: no blanks left

# CHECK
print(clean[["price_gbp", "temperature_c"]].dtypes)                         # both should be float64 (numbers)
print("Temp range:", clean["temperature_c"].min(), "to", clean["temperature_c"].max())   # should look like London
print("Weather:", clean["weather"].value_counts().to_dict())               # 4 types, no blanks
print(pd.DataFrame(log, columns=["step", "rows_affected"]).tail(4))        # last 4 log entries

price_gbp        float64
temperature_c    float64
dtype: object
Temp range: -2.0 to 25.6
Weather: {'Cloudy': 5184, 'Rainy': 4128, 'Sunny': 3456, 'Snow': 312}
                                                 step  rows_affected
8                     Standardised stockout to Yes/No            261
9                           Removed £ sign from price           1308
10  Fixed temperature (Fahrenheit, text, blanks) u...           1019
11  Filled blank weather using most common weather...            392


In [40]:
print(pd.DataFrame(log, columns=["step", "rows_affected"]))   # show the whole cleaning log

                                                 step  rows_affected
0                        Removed exact duplicate rows            262
1                 Corrected day_of_week to match date             65
2                                 Sorted rows by date          13080
3                     Standardised item names to menu            654
4                               Standardised category            523
5                       Standardised weather spelling            763
6                  Standardised 'no festival' to None          11952
7                  Filled blank local_event with None          12504
8                     Standardised stockout to Yes/No            261
9                           Removed £ sign from price           1308
10  Fixed temperature (Fahrenheit, text, blanks) u...           1019
11  Filled blank weather using most common weather...            392


In [41]:
# FIX 15: negative numbers, remove the typed minus sign
neg = (clean["sold_qty"] < 0) | (clean["wasted_qty"] < 0)          # find rows with a minus in either column
clean["sold_qty"] = clean["sold_qty"].abs()                         # abs() = remove the minus: -39 becomes 39
clean["wasted_qty"] = clean["wasted_qty"].abs()                     # same for waste
log.append(("Removed minus sign from negative quantities", neg.sum()))

# FIX 16: impossible sales, sold more than was prepared
impossible = clean["sold_qty"] > clean["prepared_qty"]              # can't sell 670 if you made 67
clean.loc[impossible, "sold_qty"] = float("nan")                    # blank it out, don't guess the real number
log.append(("Blanked impossible sold_qty (more than prepared)", impossible.sum()))

# FIX 17a: fill blank sales for FRESH items: sold = prepared - wasted
fresh = clean["shelf_life"] == "Same day"                           # freshly made items only
fill = fresh & clean["sold_qty"].isna()                             # fresh rows with a blank sold_qty
clean.loc[fill, "sold_qty"] = clean.loc[fill, "prepared_qty"] - clean.loc[fill, "wasted_qty"]  # work it out
log.append(("Filled blank sold_qty for fresh items (prepared - wasted)", fill.sum()))

# FIX 17b: drop rows we can't rebuild (bottles with blank sales)
drop = clean["sold_qty"].isna()                                     # anything still blank
clean = clean[~drop].reset_index(drop=True)                         # keep only rows that are NOT blank
log.append(("Dropped bottle rows with unknown sold_qty", drop.sum()))
clean["sold_qty"] = clean["sold_qty"].astype(int)                   # whole numbers again: no half dosas

# FIX 18: recount waste for fresh items: wasted = prepared - sold
fresh = clean["shelf_life"] == "Same day"                           # recalculate, because we dropped rows
correct_waste = clean["prepared_qty"] - clean["sold_qty"]           # what waste SHOULD be
bad = fresh & (clean["wasted_qty"] != correct_waste)                # rows that don't add up
clean.loc[bad, "wasted_qty"] = correct_waste[bad]                   # trust the till, fix the waste
log.append(("Corrected wasted_qty to prepared - sold (fresh items)", bad.sum()))

# CHECK
print(clean[["prepared_qty", "sold_qty", "wasted_qty"]].describe().round(1))   # min should be 0, no crazy max
print("Shape:", clean.shape)                                                   # rows left
print(pd.DataFrame(log, columns=["step", "rows_affected"]).tail(6))            # last 6 log entries

       prepared_qty  sold_qty  wasted_qty
count       13046.0   13046.0     13046.0
mean           19.7      15.8         1.0
std            17.0       9.5         1.5
min             4.0       3.0         0.0
25%            10.0       9.0         0.0
50%            15.0      14.0         0.0
75%            23.0      20.0         1.0
max           258.0     117.0        15.0
Shape: (13046, 18)
                                                 step  rows_affected
11  Filled blank weather using most common weather...            392
12        Removed minus sign from negative quantities             83
13   Blanked impossible sold_qty (more than prepared)             39
14  Filled blank sold_qty for fresh items (prepare...            201
15          Dropped bottle rows with unknown sold_qty             34
16  Corrected wasted_qty to prepared - sold (fresh...            128


In [42]:
# STAGE E: validation checklist, every rule must be True
checks = {
    "No duplicate rows": clean.duplicated().sum() == 0,                                  # no copies
    "record_id is unique": clean["record_id"].is_unique,                                 # every ID once
    "No blanks anywhere": clean.isna().sum().sum() == 0,                                 # zero missing values
    "Rows in date order": clean["date"].is_monotonic_increasing,                         # oldest to newest
    "Dates within Jan 2025 to Jun 2026": clean["date"].between("2025-01-01", "2026-06-30").all(),
    "day_of_week matches date": (clean["date"].dt.day_name() == clean["day_of_week"]).all(),
    "24 menu items only": set(clean["item_name"]) == set(MENU),                          # same as the official menu
    "Category is Food or Drink": set(clean["category"]) == {"Food", "Drink"},
    "Weather has 4 valid types": set(clean["weather"]) <= {"Sunny", "Cloudy", "Rainy", "Snow"},
    "Stockout is Yes or No": set(clean["stockout"]) == {"Yes", "No"},
    "One price per item": clean.groupby("item_name")["price_gbp"].nunique().max() == 1,
    "One temperature per day": clean.groupby("date")["temperature_c"].nunique().max() == 1,
    "Temperature in London range": clean["temperature_c"].between(-10, 35).all(),
    "No negative quantities": (clean[["prepared_qty", "sold_qty", "wasted_qty"]] >= 0).all().all(),
    "Sold never more than prepared": (clean["sold_qty"] <= clean["prepared_qty"]).all(),
    "Fresh waste adds up": ((clean["prepared_qty"] - clean["sold_qty"] == clean["wasted_qty"])
                            | (clean["shelf_life"] != "Same day")).all(),               # bottles are excused
    "Stockout Yes means sold out": ((clean["stockout"] == "No")
                                    | (clean["sold_qty"] == clean["prepared_qty"])).all(),
    "At least 1,000 rows": len(clean) >= 1000,                                           # course requirement
}

for name, passed in checks.items():                          # go through every rule
    print(("✅" if passed else "❌"), name)                    # tick or cross
assert all(checks.values()), "Some checks failed!"           # tripwire: stop if ANY rule fails

# SAVE: clean data + cleaning log
clean.to_csv("uxbridge_lounge_sales_clean.csv", index=False)                         # the clean dataset
pd.DataFrame(log, columns=["step", "rows_affected"]).to_csv("cleaning_log.csv", index=False)  # the evidence
print(f"Saved {len(clean):,} clean rows and {len(log)} log steps")

from google.colab import files                               # Colab's download helper
files.download("uxbridge_lounge_sales_clean.csv")            # download to your laptop
files.download("cleaning_log.csv")

✅ No duplicate rows
✅ record_id is unique
✅ No blanks anywhere
✅ Rows in date order
✅ Dates within Jan 2025 to Jun 2026
✅ day_of_week matches date
✅ 24 menu items only
✅ Category is Food or Drink
✅ Weather has 4 valid types
✅ Stockout is Yes or No
✅ One price per item
✅ One temperature per day
✅ Temperature in London range
✅ No negative quantities
✅ Sold never more than prepared
✅ Fresh waste adds up
✅ Stockout Yes means sold out
✅ At least 1,000 rows
Saved 13,046 clean rows and 17 log steps


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>